# Create multi-agent research tool using Autogen

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET



In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o")

Implement an arXiv search function

In [3]:
def arxiv_search_tool(query: str, max_results: int) -> str:
    """
    Search for papers on arXiv based on the given query.

    Args:
        query (str): The search query.
        max_results (int): The maximum number of results to return.

    Returns:
        str: A list of the top results.
    """
    # Implementation for arXiv search would go here
    url = f'http://export.arxiv.org/api/query'
    response = requests.get(url, params={'search_query': query, 'max_results': max_results})

    if response.status_code == 200:
        root = ET.fromstring(response.content)
        entries = root.findall('{http://www.w3.org/2005/Atom}entry')
        results = []
        for entry in entries:
            title = entry.find('{http://www.w3.org/2005/Atom}title').text
            summary = entry.find('{http://www.w3.org/2005/Atom}summary').text
            results.append(f"Title: {title}\nSummary: {summary}\n")
        return "\n".join(results)
    else:
        return f"Error: Unable to fetch results from arXiv. Status code: {response.status_code}"
    

In [6]:
result = arxiv_search_tool("AI in healthcare", 3)
print(result)

Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Summary: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models. We present a broadly accessible overview of the development life cycle of clinical AI models that is general enough to be adapted to most machine learning projects, and then give an in-depth case study of the development process of a deep learning based system to detect aortic aneurysms in Computed Tomography (CT) exams. We hope other healthcare institutions and clinical practitioners find the insights we share about the development process useful in informing their own model development efforts and to increase the likelihood of successful deployment and integration of AI in healthcare.

Title: Fait

Create a Topic refinement agent

In [5]:
agent_description = "Refines a broad research idea into a specific topic."
SYSTEM_MESSAGE = """Role: You are a research topic refinement agent.
Task: Given a generic idea, output one refined research topic.
Output_format: 
  Refined Research Topic: <refined topic>
  End with TASK_COMPLETED_TOPIC_AGENT
"""
tools = [arxiv_search_tool]
topicAgent = AssistantAgent(name="TopicAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            tools = tools,
                            model_client = client
                            )

topic_result = await topicAgent.run(task="Newton's laws of motion")
print(topic_result.messages[-1].content)


Refined Research Topic: "Analyzing the Application of Newton's Laws of Motion in Modern Engineering Systems and Their Role in Advancing Autonomous Vehicle Technology"

End with TASK_COMPLETED_TOPIC_AGENT


Create a Paper discovery agent

In [7]:
agent_description = "Searches for relevant papers based on a research topic."
SYSTEM_MESSAGE = """Role: You are a research paper search agent.
Task: Given a research topic, search for relevant papers and return the top results.
"""
tools = [arxiv_search_tool]
paperAgent = AssistantAgent(name="PaperAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            tools = tools,
                            model_client = client
                            )

task = "AI in healthcare"
paper_result = await paperAgent.run(task=task)
print(paper_result.messages[-1].content)

Title: An Overview and Case Study of the Clinical AI Model Development Life Cycle for Healthcare Systems
Summary: Healthcare is one of the most promising areas for machine learning models to make a positive impact. However, successful adoption of AI-based systems in healthcare depends on engaging and educating stakeholders from diverse backgrounds about the development process of AI models. We present a broadly accessible overview of the development life cycle of clinical AI models that is general enough to be adapted to most machine learning projects, and then give an in-depth case study of the development process of a deep learning based system to detect aortic aneurysms in Computed Tomography (CT) exams. We hope other healthcare institutions and clinical practitioners find the insights we share about the development process useful in informing their own model development efforts and to increase the likelihood of successful deployment and integration of AI in healthcare.

Title: Fait

In [ ]:
insight_agent = "Reads the papers and summarizes key ideas and insights."
SYSTEM_MESSAGE = """Role: You are an insight synthesizer agent.
Task: your job is to extract meaningful insights and highlight the main trends or findings across multiple papers. You should provide concise summary in 3 to 5 bullet points.
OUTPUT FORMAT:
Key Insights:
- <insight 1>
- <insight 2>
"""
insightAgent = AssistantAgent(name="InsightAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )

insight_result = await insightAgent.run(task=paper_result.messages[-1].content)
print(insight_result.messages[-1].content)

Key Insights:

- Successful adoption of AI in healthcare requires education and engagement of diverse stakeholders, highlighting the importance of a comprehensible AI model development life cycle and deployment case studies, such as deep learning systems for detecting medical conditions like aortic aneurysms.

- Perceived authority of AI in predictive roles influences human decision-making significantly, often leading individuals to alter their reasoning about future behaviors, as observed in a study illustrating that reliance on AI predictions can result in irrational decision outcomes and financial losses.

- Generative AI models are revolutionizing information access systems by enabling advanced content creation (information generation) and methodical integration of existing data (information synthesis). These capabilities enhance user experiences and support the provision of accurate responses, showcasing the foundational transformation these models bring to IA technologies.

- Pri

In [ ]:
report_agent = "Creates a comprehensive report based on the insights."
SYSTEM_MESSAGE = """Role: You are a report creator agent.
Task: your job is to create a comprehensive report based on the insights extracted from multiple papers. You should provide a well-structured summary that highlights the main findings and their implications.
OUTPUT FORMAT:
##Research Report
### Introduction
...
### Key Findings
...
### Discussion
...
### Conclusion
...
"""
reportAgent = AssistantAgent(name="ReportAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )

report_result = await reportAgent.run(task=insight_result.messages[-1].content)
print(report_result.messages[-1].content)

## Research Report
### Introduction
The advancement of artificial intelligence (AI) in various sectors promises transformative capabilities, particularly in healthcare. This report synthesizes insights from recent studies to highlight the essential components for successful AI adoption, the impact of AI on decision-making, the evolution of information access through generative AI models, privacy considerations in healthcare machine learning (ML) applications, and the importance of robust security architectures.

### Key Findings
1. **Adoption of AI in Healthcare**:
   Successful integration of AI into healthcare hinges on educating and engaging diverse stakeholders. A comprehensible development life cycle for AI models is essential, as evidenced by the efficacy of deep learning systems in detecting conditions like aortic aneurysms.

2. **AI’s Influence on Decision-Making**:
   The perceived authority of AI in predictive analytics significantly affects human decision-making processes. S

In [ ]:
report_agent = "Creates research gap analysis."
SYSTEM_MESSAGE = """Role: You are a report gap analysis agent.
Task: Identify 2–3 research gaps and 2–3 future directions.

    OUTPUT FORMAT:
    Research Gaps:
    - <Gap 1>
    - <Gap 2>

    Future Directions:
    - <Direction 1>
    - <Direction 2>

    FINAL REPORT
"""
reportAgent = AssistantAgent(name="ReportAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )

report_result = await reportAgent.run(task=insight_result.messages[-1].content)
print(report_result.messages[-1].content)